In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import peft
import transformers

print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

PEFT: 0.19.1
Transformers: 5.12.1


In [3]:
# !pip install -q transformers peft accelerate bitsandbytes sentencepiece

In [4]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import PeftModel

from IPython.display import Markdown, display

# **Inference Testing**

In [5]:
# Configurations
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_MODEL = "igmoiiz/math-tutor-qwen-lora"

In [6]:
# Checking For GPU
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [7]:
# Loading the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
# Loading the BASE MODEL
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
# Loading the Fine Tuned Adapter from igmoiiz Hugging Face
model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

model.eval()

print("Math Tutor Loaded Successfully!")

adapter_config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

Math Tutor Loaded Successfully!


In [10]:
# Counting Parameters
trainable = 0
total = 0

for _, param in model.named_parameters():
    total += param.numel()
    if param.requires_grad:
        trainable += param.numel()

print(f"Total Parameters: {total:,}")
print(f"Trainable Parameters: {trainable:,}")

Total Parameters: 1,548,072,448
Trainable Parameters: 0


In [11]:
# Inference Function
def ask(question, max_new_tokens=256):
    messages = [
        {
            "role": "system", "content":
            """You are an expert math tutor.

            Solve the problem step by step.

            Always end your response with exactly:

            Final Answer: <number>

            Example:
            Final Answer: 18
            """
        },
        {
          "role": "user",
          "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [12]:
# First Question Test
question = """
A train travels 80 km in 2 hours.
What is its average speed?
"""

response = ask(question)

print(display(Markdown(response)))

To find the average speed of a train that travels 80 kilometers in 2 hours, we use the formula for average speed:

\[ \text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}} \]

Given:
- Total Distance = 80 km
- Total Time = 2 hours

Plugging these values into the formula gives us:

\[ \text{Average Speed} = \frac{80 \text{ km}}{2 \text{ hours}} \]

Now, perform the division:

\[ \text{Average Speed} = 40 \text{ km/hour} \]

Final Answer: 40 km/hour

None


In [13]:
# Hard Question Test
print(
    ask(
        """
Sarah bought 8 notebooks.

Each notebook costs $4.

She also bought 3 pens costing $2 each.

How much money did she spend altogether?
"""
    )
)

Step 1: Calculate the total cost of the notebooks.
- Sarah bought 8 notebooks at $4 each.
- Total cost for notebooks = Number of notebooks × Cost per notebook
- Total cost for notebooks = 8 × $4 = $32

Step 2: Calculate the total cost of the pens.
- Sarah bought 3 pens at $2 each.
- Total cost for pens = Number of pens × Cost per pen
- Total cost for pens = 3 × $2 = $6

Step 3: Add the total cost of the notebooks and the pens to find the total amount spent.
- Total amount spent = Total cost for notebooks + Total cost for pens
- Total amount spent = $32 + $6 = $38

Final Answer: $38


In [14]:
# Multi Step Reasoning
print(
    ask(
        """
Ali had 50 marbles.

He gave 18 to his friend.

Then he bought 27 more.

How many marbles does he have now?
"""
    )
)

Step 1: Ali initially had 50 marbles.
Step 2: He gave away 18 marbles, so we subtract that from the initial amount:
   50 - 18 = 32 marbles remaining.
Step 3: Then he bought 27 more marbles. We add those to the current total:
   32 + 27 = 59 marbles in total.
Final Answer: 59


In [15]:
# Fraction Problem
print(
    ask(
        """
A pizza is cut into 8 equal slices.

Ahmed eats 3 slices.

What fraction of the pizza remains?
"""
    )
)

Step 1: Determine the total number of slices.
The pizza is cut into 8 equal slices.

Step 2: Calculate how many slices Ahmed ate.
Ahmed ate 3 slices.

Step 3: Subtract the number of slices eaten from the total number of slices to find out how many slices remain.
Total slices - Slices eaten = Remaining slices
8 - 3 = 5 remaining slices

Step 4: Express the remaining slices as a fraction of the whole pizza.
Since there were originally 8 slices, we express the remaining 5 slices as a fraction of the whole pizza.

Remaining slices / Total slices = Fraction of pizza left
5/8

Final Answer: 5/8


In [16]:
# Geometry Problem
print(
    ask(
        """
A rectangle has length 15 cm and width 8 cm.

Find its area.
"""
    )
)

To find the area of a rectangle, you use the formula:

\[ \text{Area} = \text{length} \times \text{width} \]

Given that the length is 15 cm and the width is 8 cm, we can substitute these values into the formula:

\[ \text{Area} = 15 \, \text{cm} \times 8 \, \text{cm} \]

Now, let's perform the multiplication:

\[ \text{Area} = 120 \, \text{cm}^2 \]

So, the area of the rectangle is \( 120 \, \text{cm}^2 \).

Final Answer: 120
